# Daily Climate-Health Models: Explanatory and Predictive

Two tracks, sharing the same daily panel dataset.

**Track A — Explanatory (DLNM, no AR)**
Quasi-Poisson GLM with seasonal spline + DOW dummies + DLNM cross-basis.
Tests whether climate affects daily diarrheal cases, independent of AR structure.
Fits main effect and infrastructure-quality interaction.

**Track B — Predictive (multi-horizon)**
Evaluates climate's contribution at forecast horizons 1, 7, 14, 21, 28 days.
Compares AR+Seasonal+Climate vs AR+Seasonal vs Climate-only (no AR) vs Seasonal-only.
The climate-only row is the key early-warning system benchmark.

**Input:** `{OUT_ROOT}/modeling/{NETWORK}_{HSA_MODE}_daily_modeling_dataset_{BOUNDARY_VERSION}.csv`


In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
# Choose which HSA boundary bundle to use. Must match a bundle from HSA_FINAL.ipynb.
#   'v6'  — original greedy algorithm (no post-selection corrections)
#   'v7'  — + anchor upgrade/demotion + major-orphan promotion
#   'v8'  — + satellite bubble boundaries
BOUNDARY_VERSION = "v8"         # change as needed

NETWORK          = "INF"        # change as needed (INF or NCD)
DISEASE_FOCUS    = "diarrheal"  # group focus; resolved to canonical label + slug
HSA_MODE         = "footprint"  # change as needed
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
# Setup
import sys
from pathlib import Path
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Import cross-basis functions from the local dlnm/ package
sys.path.insert(0, str(Path(".")))
from dlnm.dlnm_crossbasis import ns_basis, build_crossbasis, cumulative_rr
from disease_focus import canonical_group, daily_outcome_col
OUTCOME_COL = daily_outcome_col(canonical_group(NETWORK, DISEASE_FOCUS))

OUT_ROOT  = Path(os.environ.get("HSA_OUT_DIR", os.environ.get("PIPELINE_OUT_DIR", "out")))
DATA_FILE = OUT_ROOT / "modeling" / f"{NETWORK}_{HSA_MODE}_daily_modeling_dataset_{BOUNDARY_VERSION}.csv"
OUT_DIR   = OUT_ROOT / "modeling" / f"daily_models_{BOUNDARY_VERSION}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries loaded")
print(f"Boundary version: {BOUNDARY_VERSION}")
print(f"Network:          {NETWORK}")
print(f"Data file: {DATA_FILE}")
print(f"Output: {OUT_DIR.resolve()}")


In [ ]:
# Load data
df = pd.read_csv(DATA_FILE, parse_dates=["date"])
df = df.sort_values(["hsa_id", "date"]).reset_index(drop=True)
print(f"Loaded: {df.shape}  HSAs={df['hsa_id'].nunique()}  "
      f"dates={df['date'].dt.date.min()} to {df['date'].dt.date.max()}")

# HSAs with mean > 1 case/day (avoid sparse fitting)
hsa_means = df.groupby("hsa_id")[OUTCOME_COL].mean()
low_count_hsas = hsa_means[hsa_means < 1.0].index.tolist()
if low_count_hsas:
    print(f"Excluding {len(low_count_hsas)} low-count HSAs: {low_count_hsas}")

df_full = df[~df["hsa_id"].isin(low_count_hsas)].copy().reset_index(drop=True)
print(f"Analysis dataset: {df_full.shape}  HSAs={df_full['hsa_id'].nunique()}")


In [ ]:
# ─── Base predictors (shared across both tracks) ─────────────────────────────

outcome    = df_full[OUTCOME_COL].values.astype(float)
infra      = df_full["infra_quality"].fillna(df_full["infra_quality"].mean()).values
infra_c    = infra - infra.mean()

# HSA fixed effects
hsa_dummies = pd.get_dummies(df_full["hsa_id"], drop_first=True, dtype=float)

# Seasonal spline: ~5 knots per year of study
n_days = int(df_full["day_of_study"].max()) + 1
n_int  = max(5, n_days // 90)          # roughly one interior knot per quarter
spline_t, spline_knots = ns_basis(df_full["day_of_study"].values, n_interior_knots=n_int)

# DOW dummies (reference = Monday/0)
dow_dummies = pd.get_dummies(df_full["day_of_week"],
                             prefix="dow", drop_first=True, dtype=float)


# Calendar indicators (validated against raw attendance data):
#   is_friday    — Friday 54% of uniform attendance; captured by DOW dummy but
#                  included as explicit column for easy interpretation
#   is_ramadan   — 27% reduction in daily visits during Ramadan
#   is_eid_fitr  — multi-day drop after Ramadan (Eid al-Fitr)
#   is_eid_adha  — sharp 1-2 day drop for Eid al-Adha
# Note: Saturday is NOT a low-attendance day in Jordan (100% of uniform).
#       Other secular holidays show inconsistent signals and are excluded.
for col in ["is_ramadan", "is_eid_fitr", "is_eid_adha"]:
    if col not in df_full.columns:
        df_full[col] = 0
        print(f"  WARNING: {col} not found, set to 0")

calendar_X = np.c_[
    df_full["is_ramadan"].values.astype(float),
    df_full["is_eid_fitr"].values.astype(float),
    df_full["is_eid_adha"].values.astype(float),
]

base_X = np.c_[
    hsa_dummies.values,
    spline_t,
    dow_dummies.values,   # includes Friday dummy (day_of_week==4)
    calendar_X,
]
print(f"Base design matrix: {base_X.shape}")
print(f"  HSA FE:     {hsa_dummies.shape[1]}")
print(f"  Spline:     {spline_t.shape[1]} (df={n_int+1})")
print(f"  DOW:        {dow_dummies.shape[1]} (Fri= {int((df_full['day_of_week']==4).sum())} days)")
print(f"  Ramadan:    {int(df_full['is_ramadan'].sum())} HSA-days")
print(f"  Eid Fitr:   {int(df_full['is_eid_fitr'].sum())} HSA-days")
print(f"  Eid Adha:   {int(df_full['is_eid_adha'].sum())} HSA-days")


## Track A — Explanatory DLNM

In [ ]:
# ─── Track A: precipitation cross-basis ──────────────────────────────────────
# Lag 0 = same-day precip; lag 1-14 = prior days
lag_vals = np.arange(0, 15, dtype=float)   # 0..14

PRECIP_VARS = ["P_precip"] + [f"P_precip_lag{k}" for k in range(1, 15)]
Q_precip = df_full[PRECIP_VARS].values  # shape (n_obs, 15)

# Knot strategy: 80th pct of non-zero values
all_p = Q_precip.flatten()
nonzero_p = all_p[all_p > 0]
zero_frac  = (all_p == 0).mean()
int_knot_p = np.percentile(nonzero_p, 80) if zero_frac > 0.3 else np.percentile(all_p, 50)
exp_all_knots_p = np.array([all_p.min(), int_knot_p, all_p.max()])

lag_int_knots = np.array([3.0, 7.0])
lag_all_knots = np.array([lag_vals[0], lag_int_knots[0], lag_int_knots[1], lag_vals[-1]])

CB_precip, _, cb_meta = build_crossbasis(
    Q_precip,
    exp_n_int=1,
    lag_n_int=2,
    exp_all_knots=exp_all_knots_p,
    lag_all_knots=lag_all_knots,
    lag_values=lag_vals,
)
print(f"Precipitation cross-basis: {CB_precip.shape}")
print(f"  Zero fraction: {zero_frac:.1%}")
print(f"  Interior knot: {int_knot_p:.2f} mm")
print(f"  Lag knots: {lag_all_knots}")


In [ ]:
# ─── Fit quasi-Poisson models ─────────────────────────────────────────────────

def fit_qp(X, y):
    m = sm.GLM(y, sm.add_constant(X.astype(float), has_constant="add"),
               family=sm.families.Poisson())
    return m.fit(scale="X2")

def ftest(res_r, res_f):
    dD  = res_r.deviance - res_f.deviance
    ddf = int(round(res_r.df_resid - res_f.df_resid))
    if ddf <= 0:
        return np.nan, np.nan
    F = (dD / ddf) / res_f.scale
    p = 1 - stats.f.cdf(F, ddf, res_f.df_resid)
    return float(F), float(p)

CB_x_infra = CB_precip * infra_c[:, None]

print("Fitting base model...")
res_base  = fit_qp(pd.DataFrame(base_X), outcome)

print("Fitting main-effect model (base + CB_precip)...")
res_main  = fit_qp(np.c_[base_X, CB_precip], outcome)

print("Fitting interaction model (base + CB_precip + CB×infra)...")
res_inter = fit_qp(np.c_[base_X, CB_precip, CB_x_infra], outcome)

F_main, p_main = ftest(res_base,  res_main)
F_int,  p_int  = ftest(res_main,  res_inter)

print()
print("═" * 55)
print("Track A: Precipitation DLNM results")
print("═" * 55)
print(f"  φ (dispersion):              {res_inter.scale:.2f}")
print(f"  Main effect F-test:          F={F_main:.3f}  p={p_main:.4f}")
print(f"  Sanitation interaction:      F={F_int:.3f}   p={p_int:.4f}")


In [ ]:
# ─── Cumulative RR plot ───────────────────────────────────────────────────────
n_cb = CB_precip.shape[1]
n_base_const = base_X.shape[1] + 1   # +1 for constant added by fit_qp
coef_cb  = np.asarray(res_inter.params)[n_base_const : n_base_const + n_cb]
vcov_cb  = np.asarray(res_inter.cov_params())[
    n_base_const : n_base_const + n_cb,
    n_base_const : n_base_const + n_cb,
]

# Reference: median non-zero precip
ref_val = np.percentile(nonzero_p, 50)

# Evaluate at percentiles of observed precip
eval_pts = np.percentile(nonzero_p, np.arange(5, 96, 5))

cum_log_rr, cum_se = cumulative_rr(coef_cb, vcov_cb, cb_meta, eval_pts, reference_exp=ref_val)
cum_rr = np.exp(cum_log_rr)
cum_lo = np.exp(cum_log_rr - 1.96 * cum_se)
cum_hi = np.exp(cum_log_rr + 1.96 * cum_se)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(eval_pts, cum_rr, color="steelblue", lw=2, label="Cum. RR")
ax.fill_between(eval_pts, cum_lo, cum_hi, alpha=0.2, color="steelblue")
ax.axhline(1.0, color="black", lw=0.8, ls="--")
ax.axvline(ref_val, color="gray", lw=0.8, ls="--", label=f"ref={ref_val:.1f}mm")
ax.set_xlabel("Daily precipitation (mm)")
ax.set_ylabel("Cumulative RR (lags 0–14)")
ax.set_title("Track A: Precipitation effect on daily diarrheal counts")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "trackA_precip_cumRR.png", dpi=150)
plt.close()
print("Saved: trackA_precip_cumRR.png")


In [ ]:
# ─── Track A: precipitation RR stratified by sanitation (low vs high quartile) ─
# Effective cross-basis coefficients at centered sanitation level v = infra - mean:
#   beta_eff(v) = beta_main + v * beta_interaction
# Reported as cumulative RR (lags 0-14) at a high-rainfall level vs the plot ref.
n_cb2 = CB_precip.shape[1]
nbc   = base_X.shape[1] + 1
_pm   = np.asarray(res_inter.params)
_pv   = np.asarray(res_inter.cov_params())
idx_m = np.arange(nbc,         nbc + n_cb2)
idx_i = np.arange(nbc + n_cb2, nbc + 2 * n_cb2)
beta_m, beta_i = _pm[idx_m], _pm[idx_i]
V_mm = _pv[np.ix_(idx_m, idx_m)]
V_mi = _pv[np.ix_(idx_m, idx_i)]
V_ii = _pv[np.ix_(idx_i, idx_i)]

_hsa_infra = df_full.groupby("hsa_id")["infra_quality"].mean()
_q1, _q3 = _hsa_infra.quantile(0.25), _hsa_infra.quantile(0.75)
_low  = _hsa_infra[_hsa_infra <= _q1].mean()
_high = _hsa_infra[_hsa_infra >= _q3].mean()
_mean = infra.mean()

RAIN_MM = 20.0
rows = []
for label, san in [("low_sanitation_quartile", _low), ("high_sanitation_quartile", _high)]:
    v = san - _mean
    beta_eff = beta_m + v * beta_i
    V_eff    = V_mm + v * (V_mi + V_mi.T) + (v ** 2) * V_ii
    logrr, se = cumulative_rr(beta_eff, V_eff, cb_meta, np.array([RAIN_MM]), reference_exp=ref_val)
    rr = float(np.exp(logrr[0]))
    rows.append({
        "stratum": label, "mean_sanitation": round(float(san), 1),
        "rain_mm": RAIN_MM, "ref_mm": round(float(ref_val), 2),
        "cum_RR": round(rr, 3),
        "RR_lo": round(float(np.exp(logrr[0] - 1.96 * se[0])), 3),
        "RR_hi": round(float(np.exp(logrr[0] + 1.96 * se[0])), 3),
        "pct_change": round((rr - 1) * 100, 1),
    })
rr_df = pd.DataFrame(rows)
rr_df.to_csv(OUT_DIR / "trackA_precip_RR_by_sanitation.csv", index=False)
print(f"Precip cumulative RR by sanitation (at {RAIN_MM:.0f}mm vs {ref_val:.1f}mm ref):")
print(rr_df.to_string(index=False))


In [ ]:
# ─── Track A: screen additional exposures ─────────────────────────────────────

OTHER_VARS = {
    "T_mean_C":      "Daily mean temperature (°C)",
    "T_max_C":       "Daily max temperature (°C)",
    "Td_C":          "Dewpoint temperature (°C)",
    "SM1":           "Surface soil moisture (m³/m³)",
    "wind_speed_ms": "Wind speed (m/s)",
}

print(f"{'Variable':<16} {'F_main':>7} {'p_main':>8} {'F_int':>7} {'p_int':>8}  Sig")
print("-" * 60)

screen_results = []

for varname, desc in OTHER_VARS.items():
    lag_cols = [varname] + [f"{varname}_lag{k}" for k in range(1, 15)]
    if any(c not in df_full.columns for c in lag_cols):
        print(f"  {varname:<14} SKIP (columns missing)")
        continue

    Q = df_full[lag_cols].values
    all_v = Q.flatten()
    zero_f = (all_v == 0).mean()
    nonzero_v = all_v[all_v > 0]
    if len(nonzero_v) == 0:
        continue
    ik = np.percentile(nonzero_v, 80) if zero_f > 0.3 else np.percentile(all_v, 50)
    ik = np.clip(ik, all_v.min() + 1e-6, all_v.max() - 1e-6)

    CB, _, meta_v = build_crossbasis(
        Q, exp_n_int=1, lag_n_int=2,
        exp_all_knots=np.array([all_v.min(), ik, all_v.max()]),
        lag_all_knots=lag_all_knots,
        lag_values=lag_vals,
    )
    CB_xi = CB * infra_c[:, None]

    res_m = fit_qp(np.c_[base_X, CB],       outcome)
    res_i = fit_qp(np.c_[base_X, CB, CB_xi], outcome)

    Fm, pm = ftest(res_base, res_m)
    Fi, pi = ftest(res_m,    res_i)

    sig = "***" if pi < 0.001 else ("**" if pi < 0.01 else ("*" if pi < 0.05 else ""))
    print(f"  {varname:<14} {Fm:>7.3f} {pm:>8.4f} {Fi:>7.3f} {pi:>8.4f}  {sig}")
    screen_results.append(dict(variable=varname, description=desc,
                               F_main=Fm, p_main=pm, F_int=Fi, p_int=pi))

import pandas as pd
screen_df = pd.DataFrame(screen_results).sort_values("p_int")
screen_df.to_csv(OUT_DIR / "trackA_screening.csv", index=False)
print("\nSaved: trackA_screening.csv")


## Track B — Predictive (multi-horizon)

In [ ]:
# ─── Track B setup ────────────────────────────────────────────────────────────
# OLS on log(Y+0.5) for computational speed.
# For each horizon h, predict Y_{t+h} from features known at time t.
#
# Models:
#   Seasonal-only:          seasonal spline + DOW + HSA FE + Ramadan + Holiday
#   AR+Seasonal:            + AR(1-day lag) + AR(7-day lag)
#   Seasonal+Climate:       seasonal + climate cross-sum (no AR) — early-warning model
#   AR+Seasonal+Climate:    all three
#
# Climate features: sum over lags 0..14 per variable (parsimony).
# At horizon h, all lags 0..14 at time t are available (past data).

HORIZONS = [1, 7, 14, 21, 28]

# Climate feature matrix: sum of lags 0-14 for each variable (cross-sum approximation)
# This avoids refitting a full cross-basis for each horizon
CLIM_SUM_COLS = {}
for varname in ["P_precip", "T_mean_C", "Td_C", "SM1"]:
    lag_cols = [varname] + [f"{varname}_lag{k}" for k in range(1, 15)]
    if all(c in df_full.columns for c in lag_cols):
        col_sum = df_full[lag_cols].sum(axis=1)
        CLIM_SUM_COLS[varname] = col_sum

clim_matrix = np.column_stack(list(CLIM_SUM_COLS.values())) if CLIM_SUM_COLS else None
print(f"Climate summary features: {list(CLIM_SUM_COLS.keys())}")
print(f"Climate matrix shape: {clim_matrix.shape if clim_matrix is not None else 'None'}")


In [ ]:
# ─── Horizon evaluation loop ──────────────────────────────────────────────────

# Temporal split: train on first 80% of dates, test on final 20% across all HSAs
n_obs = len(df_full)
unique_dates = np.array(sorted(df_full["date"].unique()))
split_idx = int(len(unique_dates) * 0.80)
split_date = unique_dates[split_idx]
date_values = df_full["date"].values
print(f"Temporal split date: {pd.Timestamp(split_date).date()}  "
      f"({split_idx}/{len(unique_dates)} dates in training period)")

# Outcomes and AR features per HSA (shift must respect HSA boundaries)
Y_log = np.log(df_full[OUTCOME_COL].values + 0.5)

def make_ar_features(h):
    ar1 = df_full.groupby("hsa_id")[OUTCOME_COL].shift(1).fillna(0).values
    ar7 = df_full.groupby("hsa_id")[OUTCOME_COL].shift(7).fillna(0).values
    return np.c_[np.log(ar1 + 0.5), np.log(ar7 + 0.5)]

def make_future_y(h):
    return df_full.groupby("hsa_id")[OUTCOME_COL].shift(-h).values

def r2_oos(y_true, y_pred):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    if len(yt) == 0 or len(yt) != len(yp):
        return np.nan
    ss_res = np.sum((yt - yp)**2)
    ss_tot = np.sum((yt - yt.mean())**2)
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

results_B = []

for h in HORIZONS:
    Y_future = make_future_y(h)
    Y_future_log = np.log(np.where(Y_future > 0, Y_future, 0) + 0.5)
    valid = ~np.isnan(Y_future)

    ar = make_ar_features(h)
    X_seas  = base_X
    X_ar    = np.c_[base_X, ar]
    X_sclim = np.c_[base_X, clim_matrix] if clim_matrix is not None else base_X
    X_full  = np.c_[base_X, ar, clim_matrix] if clim_matrix is not None else X_ar

    train_mask = valid & (date_values < split_date)
    test_mask  = valid & (date_values >= split_date)

    row = {"horizon_days": h}

    for label, X in [
        ("Seasonal-only",            X_seas),
        ("AR+Seasonal",              X_ar),
        ("Seasonal+Climate (no AR)", X_sclim),
        ("AR+Seasonal+Climate",      X_full),
    ]:
        Xc_tr = sm.add_constant(X[train_mask], has_constant="add")
        Xc_te = sm.add_constant(X[test_mask],  has_constant="add")

        try:
            fit = sm.OLS(Y_future_log[train_mask], Xc_tr).fit()
            pred = fit.predict(Xc_te)
            r2 = r2_oos(Y_future_log[test_mask], pred)
        except Exception as e:
            print(f"  WARNING: {label} h={h}d failed: {e}")
            r2 = np.nan

        row[label] = round(r2, 4)

    # Climate contribution over AR baseline
    ar_r2   = row.get("AR+Seasonal", np.nan)
    full_r2 = row.get("AR+Seasonal+Climate", np.nan)
    row["ΔR² (climate)"] = round(full_r2 - ar_r2, 4) if not np.isnan(ar_r2 + full_r2) else np.nan

    results_B.append(row)
    print(f"h={h:2d}d  AR+Seas={ar_r2:.3f}  +Clim={full_r2:.3f}  "
          f"Clim-only={row.get('Seasonal+Climate (no AR)',np.nan):.3f}  "
          f"ΔR²={row['ΔR² (climate)']:.4f}")

results_B_df = pd.DataFrame(results_B)
results_B_df.to_csv(OUT_DIR / "trackB_horizon_results.csv", index=False)
print("\nSaved: trackB_horizon_results.csv")


In [ ]:
# ─── Track B: horizon plot ────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hors = [r["horizon_days"] for r in results_B]

ax = axes[0]
for model in ["Seasonal-only", "AR+Seasonal", "Seasonal+Climate (no AR)", "AR+Seasonal+Climate"]:
    vals = [r.get(model, np.nan) for r in results_B]
    ax.plot(hors, vals, marker="o", label=model)
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("Out-of-sample R²")
ax.set_title("Track B: Model R² by forecast horizon")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
delta = [r["ΔR² (climate)"] for r in results_B]
delta_plot = [0 if np.isnan(d) else d for d in delta]
colors = ["#9ca3af" if np.isnan(d) else ("#d62728" if d < 0 else "#2ca02c") for d in delta]
ax.bar(hors, delta_plot, color=colors, width=2)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("ΔR² from adding climate to AR+Seasonal")
ax.set_title("Track B: Incremental climate contribution by horizon")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(OUT_DIR / "trackB_horizon_R2.png", dpi=150)
plt.close()
print("Saved: trackB_horizon_R2.png")


In [ ]:
# ─── Summary table (both tracks) ─────────────────────────────────────────────
print("\n" + "═" * 70)
print("SUMMARY")
print("═" * 70)

print("\nTrack A — Explanatory (DLNM, no AR terms)")
print(f"  Precipitation main effect:  F={F_main:.3f}  p={p_main:.4f}")
print(f"  Sanitation interaction:     F={F_int:.3f}   p={p_int:.4f}")
print(f"  Dispersion φ:              {res_inter.scale:.2f}")

print("\nTrack B — Predictive (multi-horizon OLS on log scale)")
print(results_B_df.to_string(index=False))

print("\nKey finding:")
if p_int < 0.05:
    print("  Track A: Precipitation effect modified by sanitation infrastructure (p<0.05).")
    print("  Climate does affect diarrheal risk, but the magnitude depends on whether")
    print("  sanitation can buffer the exposure-to-contamination pathway.")
else:
    print("  Track A: No significant precipitation effect or sanitation interaction detected.")

valid_deltas = [r["ΔR² (climate)"] for r in results_B if not np.isnan(r["ΔR² (climate)"])]
if valid_deltas:
    max_delta = max(valid_deltas)
    best_h    = [r["horizon_days"] for r in results_B
                 if r["ΔR² (climate)"] == max_delta][0]
    if max_delta > 0:
        print(f"  Track B: Peak climate contribution at h={best_h}d (ΔR²={max_delta:.4f}).")
        print("  Climate adds predictive value where ΔR² is positive;")
        print("  AR terms usually dominate at short horizons.")
    else:
        print(f"  Track B: Climate did not improve holdout R² at any tested horizon; "
              f"least negative ΔR² was h={best_h}d (ΔR²={max_delta:.4f}).")
else:
    print("  Track B: No valid out-of-sample R² values were produced.")


In [ ]:
# AUTO_NOTEBOOK_SUMMARY_V1
from pathlib import Path
from datetime import datetime
import os
import json

NOTEBOOK_NAME = "run_climate_models_daily"
NETWORK = globals().get('NETWORK', os.environ.get('NETWORK', 'INF'))
HSA_MODE = globals().get('HSA_MODE', os.environ.get('HSA_MODE', 'footprint'))

suffix = f"{NETWORK}_{HSA_MODE}" if HSA_MODE else f"{NETWORK}"

out_root = Path(globals().get('OUT_ROOT', globals().get('OUT_DIR', os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', "out")))))
# OUT_DIR in this notebook points inside modeling/daily_models_{ver}; walk up to the pipeline root
if 'modeling' in str(out_root):
    out_root = out_root.parent.parent
summary_dir = out_root / 'textresults'
summary_dir.mkdir(parents=True, exist_ok=True)
BOUNDARY_VERSION = globals().get('BOUNDARY_VERSION', os.environ.get('BOUNDARY_VERSION', os.environ.get('PIPELINE_VERSION', 'v7')))
summary_path = summary_dir / f"{NOTEBOOK_NAME}_{suffix}_{BOUNDARY_VERSION}_results.md"

files = [p for p in out_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.png', '.pdf', '.geojson', '.parquet'}]
files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
important = files[:120]

NL = "\n"

blocks = []
blocks.append(f"# Notebook Results: {NOTEBOOK_NAME}")

meta = [
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"- Network: {NETWORK}",
    f"- HSA mode: {HSA_MODE}",
    f"- Boundary version: {BOUNDARY_VERSION}",
]
blocks.append(NL.join(meta))

file_lines = ['## Important Output Files', '']
for p in important:
    file_lines.append(f"- `{p}`")
blocks.append(NL.join(file_lines))

nb_path = Path(f"{NOTEBOOK_NAME}.ipynb")
if nb_path.exists():
    try:
        nb_data = json.loads(nb_path.read_text())
        blocks.append('## Displayed Cell Outputs')

        for idx, cell in enumerate(nb_data.get('cells', []), start=1):
            if cell.get('cell_type') != 'code':
                continue
            outputs = cell.get('outputs', []) or []
            if not outputs:
                continue

            blocks.append(f"### Cell {idx}")

            for out in outputs:
                otype = out.get('output_type')

                if otype == 'stream':
                    text = ''.join(out.get('text', [])) if isinstance(out.get('text', []), list) else str(out.get('text', ''))
                    text = text.rstrip()
                    if text:
                        blocks.append("```text" + NL + text + NL + "```")

                elif otype in ('execute_result', 'display_data'):
                    data = out.get('data', {})
                    if 'text/markdown' in data:
                        md = ''.join(data['text/markdown']) if isinstance(data['text/markdown'], list) else str(data['text/markdown'])
                        md = md.rstrip()
                        if md:
                            blocks.append(md)
                    elif 'text/plain' in data:
                        txt = ''.join(data['text/plain']) if isinstance(data['text/plain'], list) else str(data['text/plain'])
                        txt = txt.rstrip()
                        if txt:
                            blocks.append("```text" + NL + txt + NL + "```")
                    elif 'text/html' in data:
                        html = ''.join(data['text/html']) if isinstance(data['text/html'], list) else str(data['text/html'])
                        html = html.rstrip()
                        if html:
                            blocks.append("```html" + NL + html + NL + "```")
                    elif 'image/png' in data or 'image/jpeg' in data:
                        blocks.append('_[Image output omitted in text summary]_')

                elif otype == 'error':
                    tb = out.get('traceback', []) or []
                    if tb:
                        err = NL.join(str(t) for t in tb)
                    else:
                        err = f"{out.get('ename', 'Error')}: {out.get('evalue', '')}"
                    blocks.append("```text" + NL + err + NL + "```")

    except Exception as e:
        blocks.append('## Displayed Cell Outputs')
        blocks.append(f"Could not parse notebook outputs: {e}")

summary = (NL + NL).join(b for b in blocks if b and str(b).strip()) + NL
summary_path.write_text(summary)
print(f"Saved notebook summary: {summary_path}")


In [ ]:
# ─── Agentic model bundle: curated, disease-scoped ZIP for the comparison skill ───
# Packages the small subset of modeling/sensitivity/textresults outputs the downstream
# agentic skill reads, under ONE disease-named root folder plus a manifest.json and a
# human/agent-readable SKILL_CONTEXT.md, so bundles for different diseases/networks never
# collide when extracted side by side. Run this AFTER the notebook-summary cell above (so
# this run's textresults summary is fresh). Non-hardcoded: slug resolved from disease_focus.
import os, json, zipfile
from pathlib import Path
from datetime import datetime, timezone
from disease_focus import canonical_group, slug, daily_outcome_col

NETWORK          = globals().get('NETWORK', os.environ.get('NETWORK', 'INF'))
HSA_MODE         = globals().get('HSA_MODE', os.environ.get('HSA_MODE', 'footprint'))
BOUNDARY_VERSION = globals().get('BOUNDARY_VERSION', os.environ.get('BOUNDARY_VERSION', 'v7'))
DISEASE_FOCUS    = globals().get('DISEASE_FOCUS', os.environ.get('DISEASE_FOCUS', 'diarrheal'))
OUT_ROOT = Path(globals().get('OUT_ROOT', os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', 'out'))))
if not (OUT_ROOT / 'modeling').exists() and (OUT_ROOT.parent / 'modeling').exists():
    OUT_ROOT = OUT_ROOT.parent   # in case OUT_ROOT points at the daily_models subdir

CANON  = canonical_group(NETWORK, DISEASE_FOCUS)   # e.g. "Diarrheal Diseases"
DSLUG  = slug(CANON)                                # e.g. "diarrheal_diseases"
PREFIX = f"{NETWORK}_{HSA_MODE}"                    # e.g. "INF_footprint"
ver    = BOUNDARY_VERSION
BUNDLE_KEY = f"{NETWORK}_{HSA_MODE}_{DSLUG}_{ver}"  # disease-scoped, collision-free

ALLOWED_EXT = {'.json', '.csv', '.md', '.geojson'}  # summaries only
MAX_FILE_KB = 1500                                   # skip large raw dumps
MODES    = ['footprint','fewest','distance','governorate_tau_coverage','governorate_fewest','governorate']
VERSIONS = ['v6','v7','v8']
def _md_relevant(name):
    if NETWORK not in name: return False
    if HSA_MODE not in name and any(m in name for m in MODES if m != HSA_MODE): return False
    vpres = [v for v in VERSIONS if v in name]
    if vpres and ver not in vpres: return False
    return True

files, missing = [], []
def _add(rel):
    p = OUT_ROOT / rel
    files.append(p) if p.exists() else missing.append(rel)
def _add_glob(reldir, pat):
    d = OUT_ROOT / reldir
    if not d.exists(): missing.append(f"{reldir}/ (dir)"); return
    hits = sorted(d.glob(pat)); files.extend(hits) if hits else missing.append(f"{reldir}/{pat}")

# modeling root + result subfolders (the curated set the skill reads)
_add(f"modeling/{PREFIX}_modeling_dataset_{ver}_metadata.json")
_add(f"modeling/{PREFIX}_pipeline_run_log_{ver}.json")
C = f"modeling/results_comprehensive_{ver}"
for nm in ["model_summary.json","model_comparison_pivot.csv","climate_contribution_analysis.csv",
           "all_model_results.csv","extended_climate_analysis.json","feature_selection_info.json"]:
    _add(f"{C}/{PREFIX}_{nm}")
_add(f"modeling/results_ml_{ver}/{PREFIX}_model_comparison_summary.csv")
_add(f"modeling/results_ml_{ver}/{PREFIX}_feature_importance_rankings.csv")
_add(f"modeling/results_improved_{ver}/{PREFIX}_improved_model_comparison.csv")
_add_glob(f"modeling/results_parsimonious_{ver}", f"{PREFIX}_*")
_add_glob(f"modeling/results_anomalies_{ver}", f"{PREFIX}_anomaly_*")
# sensitivity: the 9 analysis_*_{ver} folders, scoped to THIS network+mode (no NCD bleed)
sroot = OUT_ROOT / "sensitivity"
if sroot.exists():
    for sd in sorted(sroot.glob(f"analysis_*_{ver}")):
        files.extend(p for p in sd.rglob(f"*{PREFIX}*") if p.is_file())
# textresults: written .md summaries relevant to this network/mode/version
tr = OUT_ROOT / "textresults"
if tr.exists():
    files.extend(p for p in tr.glob("*.md") if _md_relevant(p.name))
# HSA boundaries (for maps)
_add(f"{PREFIX}_hsas_{ver}.geojson")

# keep summary formats within the size cap; de-dup
sel, seen, skipped = [], set(), []
for p in files:
    if not p.is_file(): continue
    rp = p.resolve()
    if rp in seen: continue
    seen.add(rp)
    if p.suffix.lower() not in ALLOWED_EXT: continue
    if p.stat().st_size > MAX_FILE_KB * 1024: skipped.append(p); continue
    sel.append(p)
files = sel

def _rel(p): return str(p.resolve().relative_to(OUT_ROOT.resolve()))
def _cat(r):
    for k in ("sensitivity/","textresults/","modeling/"):
        if r.startswith(k): return k.rstrip("/")
    return "boundaries" if r.endswith(".geojson") else "other"
index, total = {}, 0
for p in files:
    r = _rel(p); total += p.stat().st_size; index.setdefault(_cat(r), []).append(r)
n_hsas = None
_gj = OUT_ROOT / f"{PREFIX}_hsas_{ver}.geojson"
if _gj.exists():
    try: n_hsas = len(json.load(open(_gj)).get("features", []))
    except Exception: pass

manifest = {
    "bundle_key": BUNDLE_KEY, "network": NETWORK, "hsa_mode": HSA_MODE,
    "disease_focus_input": DISEASE_FOCUS, "canonical_disease": CANON, "disease_slug": DSLUG,
    "boundary_version": ver, "outcome_col": daily_outcome_col(CANON), "n_hsas": n_hsas,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "file_count": len(files), "total_bytes": total,
    "read_first": [
        f"{BUNDLE_KEY}/SKILL_CONTEXT.md",
        f"{BUNDLE_KEY}/textresults/run_climate_models_daily_{PREFIX}_{ver}_results.md",
        f"{BUNDLE_KEY}/modeling/results_comprehensive_{ver}/{PREFIX}_model_summary.json",
        f"{BUNDLE_KEY}/modeling/results_comprehensive_{ver}/{PREFIX}_climate_contribution_analysis.csv",
    ],
    "index": {k: sorted(v) for k, v in index.items()},
}

# Human/agent-readable "how to read this bundle" guide (generated per bundle).
SKILL_CONTEXT = f"""# Bundle: {BUNDLE_KEY}

One self-contained result set from the HSA climate-health pipeline, scoped to a single
disease and delineation. Sibling archives (one per disease/network) share this exact
layout, so an agent can compare across them file by file.

## Identity
- Disease: {CANON} (slug `{DSLUG}`)
- Network / mode: {NETWORK} / {HSA_MODE}
- Boundary (algorithm) version: {ver}
- Outcome column: `{daily_outcome_col(CANON)}`
- HSA count: {n_hsas}
- Generated (UTC): {manifest['generated_at']}
- Contents: {len(files)} files, {total/1024:.0f} KB (curated summaries only)

## Start here
1. `manifest.json`: machine-readable identity plus a categorized file index and this
   `read_first` list.
2. `textresults/run_climate_models_daily_{PREFIX}_{ver}_results.md`: the daily DLNM
   (explanatory) and multi-horizon (predictive) summary in prose.
3. `modeling/results_comprehensive_{ver}/{PREFIX}_model_summary.json`: headline weekly-model metrics.
4. `modeling/results_comprehensive_{ver}/{PREFIX}_climate_contribution_analysis.csv`:
   incremental value of climate over autoregressive terms, per model family.

## Layout
- `modeling/` weekly-panel modeling outputs:
  - `results_comprehensive_{ver}/`: main set. `*_model_comparison_pivot.csv` is a
    feature_set x model R2 grid; `*_all_model_results.csv` has per feature_set/model/split
    R2/RMSE/MAE; plus `*_climate_contribution_analysis.csv`, `*_extended_climate_analysis.json`,
    `*_feature_selection_info.json`, `*_model_summary.json`.
  - `results_ml_{ver}/`: ML comparison summary and feature-importance rankings.
  - `results_improved_{ver}/`: regularized/improved model comparison.
  - `results_parsimonious_{ver}/`: AR-lag, thematic-group, best-per-group,
    feature-importance, and parsimonious-model tables.
  - `results_anomalies_{ver}/`: seasonality-adjusted anomaly models.
  - `{PREFIX}_modeling_dataset_{ver}_metadata.json`, `{PREFIX}_pipeline_run_log_{ver}.json`.
- `sensitivity/analysis_*_{ver}/`: nine robustness analyses (climate exclusion,
  heterogeneity, exclusion, extreme events, gravity, spatial autocorrelation, spatial
  comparison, variance decomposition, weight sensitivity), CSV/JSON summaries only.
- `textresults/`: prose `.md` summaries written by each pipeline notebook.
- `{PREFIX}_hsas_{ver}.geojson`: the {n_hsas} HSA boundary polygons, for maps.

## Comparing diseases across bundles
Comparison tables use stable keys, so rows/columns align across diseases:
- `*_model_comparison_pivot.csv`: rows = feature_set (ar_only, ar_temporal,
  ar_climate_temporal, ...); columns = model family (ridge, lasso, elasticnet,
  random_forest, gradient_boosting).
- `*_climate_contribution_analysis.csv`: one row per model with `ar_only_r2`,
  `ar_climate_r2`, `climate_contribution_absolute`, `climate_contribution_relative_pct`.
Read the same file from each sibling bundle and join on feature_set/model. Each bundle's
disease identity is in its `manifest.json` (`canonical_disease`, `disease_slug`).

## Caveats
- Curated summaries only: figures (.png/.pdf) and large raw dumps (>{MAX_FILE_KB} KB, e.g.
  per-unit allocation classifications) are intentionally excluded.
- All numbers reflect boundary/algorithm version {ver}; different versions are separate
  bundles and must not be mixed.
- The weekly models are a forecasting design (autoregressive terms present); the daily
  DLNM in `textresults/` is the explanatory analysis. Consult both before drawing any
  conclusion about climate effects.
"""

bundles = OUT_ROOT / "agentic_bundles"; bundles.mkdir(exist_ok=True)
zpath = bundles / f"{BUNDLE_KEY}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr(f"{BUNDLE_KEY}/manifest.json", json.dumps(manifest, indent=2))
    z.writestr(f"{BUNDLE_KEY}/SKILL_CONTEXT.md", SKILL_CONTEXT)
    for p in files:
        z.write(p, arcname=f"{BUNDLE_KEY}/{_rel(p)}")

print(f"Agentic bundle: {zpath}  ({zpath.stat().st_size/1024:.0f} KB)")
print(f"  disease={CANON} ({DSLUG})  network={NETWORK} mode={HSA_MODE} boundary={ver}  n_hsas={n_hsas}")
print(f"  {len(files)} files ({total/1024:.0f} KB) + manifest.json + SKILL_CONTEXT.md")
print("  categories: " + ", ".join(f"{k}={len(v)}" for k, v in manifest['index'].items()))
if skipped: print(f"  skipped (>{MAX_FILE_KB}KB): " + ", ".join(_rel(p) for p in skipped))
if missing: print(f"  note: {len(missing)} expected item(s) not found: {missing[:6]}")
